# R2 · 4 · Otimização: Optuna (E1) e mais sementes (E2)

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem tudo das rodadas anteriores, inclusive `semanal_lib.py`):
1. suba **`r2_lib.py`** (pasta `data_imdc_2026/RODADA2/`);
2. **crie à mão** as pastas `resultados_semanal_r2`, `resultados_uf_r2`, `cache_r2` e `avaliacao_r2` (evita pastas duplicadas quando há duas sessões abertas).

**Nada antigo é alterado.** Os modelos novos gravam em `resultados_semanal_r2/` e `resultados_uf_r2/`; as tabelas novas vão para `avaliacao_r2/` com a data e a hora no nome (`..._AAAAMMDD-HHMM.csv`) e uma cópia `..._ultimo.csv`. As tabelas da rodada anterior (`resultados_semanal_avaliacao/`) continuam como estão.

**Retomada:** se a internet ou a energia cair, reconecte e **rode tudo de novo**. Cada modelo × rodada/temporada vira um arquivo assim que termina (gravado como `.tmp` e renomeado); os treinos longos das redes gravam um checkpoint a cada 5 épocas e continuam da última época salva; o tsfresh grava uma UF por vez; o Optuna grava o estudo no Drive a cada tentativa; o Chronos ajustado fica salvo e não é refeito.

GPU (o XGBoost também usa a GPU). Parte A (GBM semanal): ~1–2 h para 40 tentativas. Parte B (MTGNN de temporada): ~3–6 h para 25 tentativas + o modelo final com 10 sementes.

**Sem vazamento:** as tentativas são avaliadas **só nas temporadas de afinação** (2018–2021); o teste só é usado pelo modelo final, avaliado no notebook R2_1. O estudo fica em `cache_r2/*.db` e é retomado: rodar de novo continua até completar `N_TENTATIVAS`.


In [ ]:
NOTEBOOK = 'r2_optuna'
PACOTES = 'optuna'

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/r2', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'dados', 'dados_uf',
                 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    if PACOTES:
        r = subprocess.run(f'pip install -q {PACOTES}', shell=True, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
sys.path.insert(0, PASTA_LOCAL)
PASTA_SEM_R2 = os.path.join(PASTA_DRIVE, 'resultados_semanal_r2')
PASTA_UF_R2 = os.path.join(PASTA_DRIVE, 'resultados_uf_r2')
PASTA_CACHE = os.path.join(PASTA_DRIVE, 'cache_r2')
PASTA_AV2 = os.path.join(PASTA_DRIVE, 'avaliacao_r2')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
ARQ_SEL = os.path.join(PASTA_DRIVE, 'resultados_semanal', 'selecao_grupos.json')
for p in (PASTA_SEM_R2, PASTA_UF_R2, PASTA_CACHE, PASTA_AV2, os.path.join(PASTA_CACHE, 'ckpt')):
    os.makedirs(p, exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S, r2_lib as R2
for m in (V, L, CM, U, S, R2):
    importlib.reload(m)
try:
    import torch
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma', '· CPUs:', os.cpu_count())
except ImportError:
    torch = None
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
RODADAS = S.rodadas(D, felipe=True)
LIM = R2.limiares(D)                        # p25/p75 de cada UF com dados até 2017 (antes da afinação)
REFAZER = False


## A · GBM semanal (XGBoost com variáveis de fase)


In [ ]:
# @title A · Optuna (retoma)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TENTATIVAS = 40
R_AF = {k: v for k, v in RODADAS.items() if S.grupo_rodada(k) == 'afinacao'}
def objetivo_gbm(trial):
    prm = dict(n_estimators=trial.suggest_int('n_estimators', 150, 900, step=50),
               max_depth=trial.suggest_int('max_depth', 3, 9),
               learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
               subsample=trial.suggest_float('subsample', 0.5, 1.0),
               colsample_bytree=trial.suggest_float('colsample_bytree', 0.4, 1.0),
               min_child_weight=trial.suggest_float('min_child_weight', 1, 50, log=True),
               reg_lambda=trial.suggest_float('reg_lambda', 0.1, 20, log=True))
    n = prm.pop('n_estimators')
    m = R2.GBMSemanalR2('opt', 'xgb', fase=True, n_estimators=n, prm=prm, lim=LIM)
    w = []
    for rod, R in R_AF.items():
        o, q = m.prever_semanal(ctx, R)
        X = S.com_verdade(S.montar('opt', rod, R, D, o, V.arruma_quantis(q)), D)
        w.append(V.wis(np.log1p(X[V.QCOLS].values), np.log1p(X.y.values)).mean())
    return float(np.mean(w))
st, salva = R2.estudo_optuna('optuna_gbm_semanal', PASTA_CACHE)
feitas = sum(t.state.name == 'COMPLETE' for t in st.trials)
if feitas < N_TENTATIVAS:
    st.optimize(objetivo_gbm, n_trials=N_TENTATIVAS - feitas, callbacks=[salva])
print('melhor (WIS log na afinação):', round(st.best_value, 4), st.best_params)


In [ ]:
# @title A2 · Modelo final com os melhores hiperparâmetros (todas as rodadas → resultados_semanal_r2)
bp = dict(st.best_params)
n = bp.pop('n_estimators')
MODELOS = [R2.GBMSemanalR2('GBM_xgb_optuna_sem', 'xgb', fase=True, n_estimators=n, prm=bp, lim=LIM)]
R2.json_grava(dict(st.best_params, valor=st.best_value), os.path.join(PASTA_CACHE, 'optuna_gbm_semanal_melhor.json'))

# rodar (retoma de onde parou; um erro não interrompe o resto)
erros = []
for m in MODELOS:
    erros += S.rodar_semanal(ctx, m, RODADAS, PASTA_SEM_R2, refazer=REFAZER)
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
print('Erros:', len(erros))
pd.DataFrame(erros, columns=['modelo', 'rodada', 'erro']).to_csv(os.path.join(PASTA_SEM_R2, f'erros_{NOTEBOOK}.csv'), index=False)
feitos = {os.path.basename(f)[:-8] for f in glob.glob(os.path.join(PASTA_SEM_R2, '*.parquet'))}
pd.DataFrame({m.nome: {r: '●' if f'{m.nome}__{r}' in feitos else '·' for r in RODADAS} for m in MODELOS}).T


## B · MTGNN de temporada (1 semente por tentativa; avaliado contra a climatologia na afinação)


In [ ]:
# @title B · Optuna (retoma)
N_TENTATIVAS_T = 25
def fabrica_mtgnn(p, sementes):
    return lambda: U.MTGNN(d=p['d'], epocas=p['epocas'], lr=p['lr'], L=p['L'], k_top=p['k_top'], beta=p['beta'],
                           sementes=sementes)
def objetivo_mtgnn(trial):
    p = dict(d=trial.suggest_categorical('d', [32, 48, 64]), epocas=trial.suggest_int('epocas', 30, 120, step=10),
             lr=trial.suggest_float('lr', 5e-4, 5e-3, log=True), L=trial.suggest_categorical('L', [52, 78, 104]),
             k_top=trial.suggest_int('k_top', 2, 8), beta=trial.suggest_float('beta', 0.02, 0.3, log=True))
    pp = trial.suggest_float('peso_pop', 0.0, 1.0)
    pasta = os.path.join(PASTA_CACHE, 'ckpt_optuna', f't{trial.number}')
    os.makedirs(pasta, exist_ok=True)
    cfg = dict(nome=f'opt{trial.number}', fabrica=fabrica_mtgnn(p, (1,)), alvo='ancora', escala=False, peso_pop=pp, cov=None)
    r = R2.R_temporada_vs_clim(ctx, R2.RedeTemporadaR2(cfg, pasta), TUNING)
    shutil.rmtree(pasta, ignore_errors=True)
    return r
stt, salva_t = R2.estudo_optuna('optuna_mtgnn_temporada', PASTA_CACHE)
feitas = sum(t.state.name == 'COMPLETE' for t in stt.trials)
if feitas < N_TENTATIVAS_T:
    stt.optimize(objetivo_mtgnn, n_trials=N_TENTATIVAS_T - feitas, callbacks=[salva_t])
print('melhor (R × climatologia na afinação):', round(stt.best_value, 4), stt.best_params)


In [ ]:
# @title B2 · E2: modelos finais com 10 sementes (Optuna e o melhor isolado anterior) → resultados_uf_r2
bp = dict(stt.best_params)
pp = bp.pop('peso_pop')
R2.json_grava(dict(stt.best_params, valor=stt.best_value), os.path.join(PASTA_CACHE, 'optuna_mtgnn_melhor.json'))
SEM10 = tuple(range(1, 11))
CK = os.path.join(PASTA_CACHE, 'ckpt')
MODELOS_T = {'MTGNN_optuna10_UF': R2.RedeTemporadaR2(dict(nome='MTGNN_optuna10_UF', fabrica=fabrica_mtgnn(bp, SEM10),
                                                           alvo='ancora', escala=False, peso_pop=pp, cov=None), CK),
             'MTGNN_pesopop10_UF': R2.RedeTemporadaR2(dict(nome='MTGNN_pesopop10_UF', fabrica=lambda: U.MTGNN(sementes=SEM10),
                                                            alvo='ancora', escala=False, peso_pop=0.5, cov=None), CK)}
TEMPORADAS = TUNING + TESTE

# rodar temporada (L.rodar: pula o que já existe; as redes retomam do último checkpoint)
erros = []
for nome, m in MODELOS_T.items():
    for s in TEMPORADAS:
        try:
            dt = L.rodar(ctx, m, s, PASTA_UF_R2, M=500, refazer=REFAZER)
            if dt is not None:
                print(f'✅ {nome:30s} {s}: {dt / 60:5.1f} min', flush=True)
        except Exception as e:
            erros.append((nome, s, repr(e)[:500]))
            print(f'❌ {nome:30s} {s}: {repr(e)[:300]}', flush=True)
            traceback.print_exc(limit=2)
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
print('Erros:', len(erros))
pd.DataFrame(erros, columns=['modelo', 'temporada', 'erro']).to_csv(os.path.join(PASTA_UF_R2, f'erros_{NOTEBOOK}.csv'), index=False)
